# Day 23 — 交易心理学与执行纪律

**核心问题**：有了策略（Day20）、边界（Day21）、仓位（Day22），怎么让自己真正执行？

**四个关键词**：损失厌恶 | 锚定效应 | 处置效应 | 恐惧贪婪

**核心原则**：决策在盘前做，不在盘中做。盘前的你是理性工程师，盘中的你是原始人。

## 一、四个心理学陷阱

交易最大的对手不是市场，是自己的大脑。四个陷阱最终导向同一个终点：**高买低卖**。

| 陷阱 | 表现 | 致命行为 | 解药 |
|------|------|----------|------|
| 损失厌恶 | 亏1万痛苦≈赚2万快乐 | 赚5%就跑，亏5%扛 | 条件单自动止损 |
| 锚定效应 | 被历史价格锁死 | 「以前XX元，现在好便宜」 | 只看MA趋势 |
| 处置效应 | 卖赚留亏 | 拔鲜花浇杂草 | 「空仓会买吗？」 |
| 恐惧贪婪 | 底部怕顶部贪 | 高买低卖循环 | 信号到了就执行 |

**共同根源**：大脑把情绪的优先级排在理性前面。

In [1]:
import pandas as pd
import numpy as np

# 损失厌恶：恒瑞医药实例
df = pd.read_csv('/tmp/恒瑞医药.csv')
df['日期'] = pd.to_datetime(df['日期'])
df = df.set_index('日期').sort_index()
d = df.loc['2025-09':'2026-01']

start_p = d['收盘'].iloc[0]
max_p = d['收盘'].max()
min_p = d['收盘'].min()
end_p = d['收盘'].iloc[-1]

print(f'恒瑞医药 2025.09-2026.01 心理历程：')
print(f'  买入价: {start_p:.2f}元')
print(f'  最高: {max_p:.2f}元 (+{(max_p/start_p-1)*100:.1f}%) → 快乐 ~{int((max_p/start_p-1)*100)}分')
print(f'  最低: {min_p:.2f}元 ({(min_p/start_p-1)*100:.1f}%) → 痛苦 ~{int(abs((min_p/start_p-1)*100)*2)}分')
print(f'  最终: {end_p:.2f}元 ({(end_p/start_p-1)*100:.1f}%)')
print()
print('💡 涨5%快乐5分，跌5%痛苦10分——同样的幅度，痛苦的权重是快乐的2倍。')
print('→ 这导致：赚一点就跑（怕回吐）、亏了死扛（不愿面对亏损）。')
print('→ 解药：条件单自动止损，不需要「下定决心」。')

恒瑞医药 2025.09-2026.01 心理历程：
  买入价: 68.15元
  最高: 72.27元 (+6.0%) → 快乐 ~6分
  最低: 56.35元 (-17.3%) → 痛苦 ~34分
  最终: 57.96元 (-15.0%)

💡 涨5%快乐5分，跌5%痛苦10分——同样的幅度，痛苦的权重是快乐的2倍。
→ 这导致：赚一点就跑（怕回吐）、亏了死扛（不愿面对亏损）。
→ 解药：条件单自动止损，不需要「下定决心」。


In [2]:
# 锚定效应：茅台 & 招商银行
for name, path in [('茅台', '/tmp/mt.csv'), ('招商银行', '/tmp/zs.csv')]:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = df.loc['2025-06':'2026-01']

    peak = d['收盘'].max()
    peak_date = d['收盘'].idxmax()
    current = d['收盘'].iloc[-1]
    ma20 = d['收盘'].rolling(20).mean().iloc[-1]
    ma60 = d['收盘'].rolling(60).mean().iloc[-1]
    trend = '多头↑' if ma20 > ma60 else '空头↓'

    print(f'{name}:')
    print(f'  高点({peak_date.date()}): {peak:.2f}元 → 现在: {current:.2f}元 ({(current/peak-1)*100:+.1f}%)')
    print(f'  MA20={ma20:.2f} MA60={ma60:.2f} → {trend}')
    if ma20 > ma60:
        print(f'  锚定者/理性者一致: 可以考虑')
    else:
        print(f'  锚定者:「{peak:.0f}跌到{current:.0f}，好便宜！」')
        print(f'  理性者:「空头排列，等趋势反转」')
    print()

print('💡 股价没有贵贱，只有方向。历史价格不构成买入理由。')

茅台:
  高点(2025-09-11): 1471.52元 → 现在: 1372.98元 (-6.7%)
  MA20=1358.83 MA60=1377.47 → 空头↓
  锚定者:「1472跌到1373，好便宜！」
  理性者:「空头排列，等趋势反转」

招商银行:
  高点(2025-07-10): 44.22元 → 现在: 37.67元 (-14.8%)
  MA20=38.20 MA60=39.76 → 空头↓
  锚定者:「44跌到38，好便宜！」
  理性者:「空头排列，等趋势反转」

💡 股价没有贵贱，只有方向。历史价格不构成买入理由。


In [3]:
# 处置效应：两只股票，必须卖一只
print('场景：同时持有海康(+8%)和比亚迪(-6%)，必须卖一只:')
print()

for name, path in [('海康威视', '/tmp/海康威视.csv'), ('比亚迪', '/tmp/byd.csv')]:
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index().iloc[-60:]
    ma20 = df['收盘'].rolling(20).mean().iloc[-1]
    ma60 = df['收盘'].rolling(60).mean().iloc[-1]
    ret = (df['收盘'].iloc[-1]/df['收盘'].iloc[0]-1)*100
    trend = '↑多头' if ma20 > ma60 else '↓空头'
    action = '✅ 应该持有' if ma20 > ma60 else '❌ 应该卖出'
    print(f'  {name}: 近60日{ret:+.1f}% | {trend} | {action}')

print()
print('80%散户卖海康（赚钱的），这是错的。')
print('解药: 问自己「如果空仓会买这只吗？」市场不关心你的成本价。')

场景：同时持有海康(+8%)和比亚迪(-6%)，必须卖一只:

  海康威视: 近60日+5.3% | ↑多头 | ✅ 应该持有
  比亚迪: 近60日-13.3% | ↓空头 | ❌ 应该卖出

80%散户卖海康（赚钱的），这是错的。
解药: 问自己「如果空仓会买这只吗？」市场不关心你的成本价。


In [4]:
# 恐惧与贪婪：比亚迪情绪过山车
df = pd.read_csv('/tmp/byd.csv')
df['日期'] = pd.to_datetime(df['日期'])
df = df.set_index('日期').sort_index()
d = df.loc['2025-07':'2026-01']

start = d['收盘'].iloc[0]
peak = d['收盘'].max()
peak_d = d['收盘'].idxmax()
trough = d['收盘'].loc[peak_d:].min()
trough_d = d['收盘'].loc[peak_d:].idxmin()
end = d['收盘'].iloc[-1]

print('比亚迪 2025.07-2026.01 情绪过山车:')
print(f'  起点 {d.index[0].date()}: {start:.2f}元')
print(f'  高点 {peak_d.date()}: {peak:.2f}元 (+{(peak/start-1)*100:.1f}%) ← 贪婪峰值')
print(f'  低点 {trough_d.date()}: {trough:.2f}元 (从高点跌{(trough/peak-1)*100:.1f}%) ← 恐惧峰值')
print(f'  终点 {d.index[-1].date()}: {end:.2f}元')
print()
print('五阶段剧本：怀疑→确认(追高)→幻想(不止损)→崩溃(割肉)→循环')
print('解药: 信号到了就执行，不管当时什么感受。')

比亚迪 2025.07-2026.01 情绪过山车:
  起点 2025-07-01: 109.12元
  高点 2025-08-29: 114.06元 (+4.5%) ← 贪婪峰值
  低点 2026-01-30: 90.89元 (从高点跌-20.3%) ← 恐惧峰值
  终点 2026-01-30: 90.89元

五阶段剧本：怀疑→确认(追高)→幻想(不止损)→崩溃(割肉)→循环
解药: 信号到了就执行，不管当时什么感受。


## 二、实战对比：有计划 vs 没计划

海康威视近120天，同一MACD策略，计划交易者 vs 情绪散户。

In [5]:
# 有计划 vs 没计划：海康威视MACD金叉
df = pd.read_csv('/tmp/海康威视.csv')
df['日期'] = pd.to_datetime(df['日期'])
df = df.set_index('日期').sort_index().iloc[-120:]

ema12 = df['收盘'].ewm(span=12, adjust=False).mean()
ema26 = df['收盘'].ewm(span=26, adjust=False).mean()
dif = ema12 - ema26
dea = dif.ewm(span=9, adjust=False).mean()

# 找MACD金叉
signals = []
for i in range(1, len(df)):
    if dif.iloc[i] > dea.iloc[i] and dif.iloc[i-1] <= dea.iloc[i-1]:
        signals.append(i)

recent = signals[-3:]
cp, cn = 100000, 100000  # 有计划 vs 没计划 初始资金

print(f'海康威视 近120天{len(signals)}次MACD金叉，取最近3次:\n')
print(f'{"信号":6s} {"买入日":12s} {"买价":>7s} {"区间":>12s} {"退出":12s} {"计划收益":>9s} {"散户收益":>9s}')
print('-'*75)

for idx, si in enumerate(recent):
    bp = df['收盘'].iloc[si]
    s10 = min(si+10, len(df)-1)
    stop_j = None
    for j in range(si+1, min(si+11, len(df))):
        if df['收盘'].iloc[j]/bp-1 <= -0.05:
            stop_j = j; break
    asi = stop_j if stop_j else s10
    ar = (df['收盘'].iloc[asi]/bp-1)*100
    er = '止损' if stop_j else '到期'
    mx = max((df['收盘'].iloc[j]/bp-1)*100 for j in range(si, min(si+11,len(df))))
    mn = min((df['收盘'].iloc[j]/bp-1)*100 for j in range(si, min(si+11,len(df))))

    # 有计划：震荡市30%仓位
    cp *= (1+ar/100*0.30)
    # 没计划：涨了就卖(只吃3%)、跌了死扛(亏2.5倍)
    nr = 3.0 if mx>5 else (ar*2.5 if ar<0 else ar)
    cn *= (1+nr/100*0.55)

    print(f'  {idx+1:2d}   {str(df.index[si].date()):12s} {bp:7.2f} {mx:+.1f}%~{mn:+.1f}% {str(df.index[asi].date()):12s} {ar:+8.1f}% {nr:+8.1f}%')

print(f'\n最终结果（3笔交易后）:')
print(f'  🤖 有计划交易者: {cp:,.0f}元 ({(cp/100000-1)*100:+.1f}%)')
print(f'  🎭 没计划散户:   {cn:,.0f}元 ({(cn/100000-1)*100:+.1f}%)')
print(f'  差距: {cp-cn:+,.0f}元')
print(f'\n💡 同一策略，只差在执行。盘前决策 vs 盘中决策 = 天壤之别。')

海康威视 近120天5次MACD金叉，取最近3次:

信号     买入日               买价           区间 退出                计划收益      散户收益
---------------------------------------------------------------------------
   1   2026-03-13     31.19 +0.7%~-6.2% 2026-03-23       -6.2%    -15.5%
   2   2026-04-09     30.01 +12.5%~+0.0% 2026-04-23       +9.1%     +3.0%
   3   2026-06-16     32.20 +7.0%~-2.5% 2026-07-01       +5.7%     +3.0%

最终结果（3笔交易后）:
  🤖 有计划交易者: 102,541元 (+2.5%)
  🎭 没计划散户:   94,536元 (-5.5%)
  差距: +8,005元

💡 同一策略，只差在执行。盘前决策 vs 盘中决策 = 天壤之别。


## 三、今日总结

### 四个陷阱 → 一个解药

```
损失厌恶 → 条件单自动止损
锚定效应 → 不看历史价格，只看MA趋势
处置效应 → 「空仓会买吗？」
恐惧贪婪 → 信号到了就执行
```

### Day20-23 四部曲

| Day | 主题 | 核心 |
|-----|------|------|
| Day20 | 六维共振策略 | 什么时候买卖 |
| Day21 | 压力测试 | 策略在什么环境有效 |
| Day22 | 仓位管理 | 用多少钱买 |
| Day23 | 交易心理学 | 怎么让自己执行 |

**有策略 + 知道边界 + 控制仓位 + 执行纪律 = 能活下来的散户。**

### 核心认知

1. 亏1万的痛苦 ≈ 赚2万的快乐——大脑天生不适合交易
2. 四个陷阱的共同终点：高买低卖
3. 盘前的你是理性工程师，盘中的你是原始人
4. 决策在盘前做，不在盘中做
5. 条件单 + 交易日志 = 执行纪律双保险
6. David Paul数据：执行计划胜率52% vs 偏离计划胜率38%

**真正的对手不是市场，是自己。赢了心理，就赢了一半。**